In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import tensorflow_hub as hub

In [6]:
df=pd.read_csv('/content/spam.csv',encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [10]:
df=df.drop(['Unnamed: 2','Unnamed: 3','Unnamed: 4'],axis=1)


KeyError: "['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'] not found in axis"

In [13]:
df=df.rename(columns={'v1':'label','v2':'Text'})
df['label_enc']=df['label'].map({'ham':0,'spam':1})
df.head()

,label,Text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [14]:
X_train,X_test,y_train,y_test=train_test_split(df['Text'],df['label_enc'],test_size=0.2,random_state=42)

In [15]:
X_train_np=X_train.to_numpy()
X_test_np=X_test.to_numpy()
y_train_np=y_train.to_numpy()
y_test_np=y_test.to_numpy()


In [18]:
avg_words_len=round(sum([len(i.split())
for i in df['Text']])/len(df['Text']))
total_words_lenght=len(set(" ".join(df['Text']).split()))

print(f"Data Loaded.Training smaples:{len(X_train_np)}")
print(f"Average words per message:{avg_words_len}")
print(f"Approximate vocabulary size:{total_words_lenght}")

Data Loaded.Training smaples:4457
Average words per message:16
Approximate vocabulary size:15686


In [27]:
def complie_and_fit(model,epochs=5):
  model.compile(
      optimizer='adam',
      loss='binary_crossentropy',
      metrics=['accuracy']
  )
  history=model.fit(
      X_train_np,
      y_train_np,
      epochs=epochs,
      validation_data=(X_test_np,y_test_np)

  )
  return history
def get_metrics(model,X,y):
  y_pred=np.round(model.predict(X))
  return {
      'accuracy':accuracy_score(y,y_pred),
      'precision':precision_score(y,y_pred),
      'recall':recall_score(y,y_pred),
      'f1-score':f1_score(y,y_pred)

  }

In [23]:
from tensorflow.keras.layers import TextVectorization
text_vec=TextVectorization(
    max_tokens=total_words_lenght,
    standardize='lower_and_strip_punctuation',
    output_mode='int',
    output_sequence_length=avg_words_len
)
text_vec.adapt(X_train_np)

In [25]:
import tensorflow as tf

input_layer=layers.Input(shape=(1,),dtype=tf.string)
x=text_vec(input_layer)
x=layers.Embedding(input_dim=total_words_lenght,output_dim=128)(x)
x=layers.Bidirectional(layers.LSTM(64,return_sequences=True))(x)
x=layers.Bidirectional(layers.LSTM(64))(x)
x=layers.Flatten()(x)
x=layers.Dropout(0.1)(x)
x=layers.Dense(32,activation='relu')(x)
x=layers.Dense(1,activation='sigmoid')(x)

model_2=keras.Model(input_layer,x,name="BiLSTM_Model")
history_2=complie_and_fit(model_2)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 19s 75ms/step - accuracy: 0.9441 - loss: 0.1669 - val_accuracy: 0.9740 - val_loss: 0.0944
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.9886 - loss: 0.0425 - val_accuracy: 0.9767 - val_loss: 0.0725
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9971 - loss: 0.0139 - val_accuracy: 0.9758 - val_loss: 0.0767
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.9996 - loss: 0.0018 - val_accuracy: 0.9686 - val_loss: 0.1631
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9982 - loss: 0.0069 - val_accuracy: 0.9794 - val_loss: 0.1041


In [29]:
results ={
    'Bi-LSTM':get_metrics(model_2,X_test_np,y_test_np)

}
results_df=pd.DataFrame(results).transpose()
print("Performance Table")
print(results_df)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
Performance Table
         accuracy  precision    recall  f1-score
Bi-LSTM  0.979372   0.943662  0.899329  0.920962
